# UAVDT Notebook 04 v2 — Metric Calibration and 3D Cuboids from GT BEV Tracks

This version is designed to run after **Notebook 03A** (`GT annotations → BEV tracks`). It avoids the previous state/order bug by defining defaults inside the cells that need them, and it searches for the GT-based `vehicle_tracks_bev.csv` first.

Expected inputs:
- Notebook 02 homography JSON in `work/notebook_02_bev_homography/`
- Notebook 03A tracks CSV in `work/notebook_03a_gt_annotations_bev_tracks/vehicle_tracks_bev.csv`

Outputs:
- `metric_calibration.json`
- `vehicle_tracks_metric.csv`
- `vehicle_cuboids_metric.csv`
- plots, OBJ, and dynamic scene JSON


In [ ]:
#@title 1. Set local project paths
from pathlib import Path
import json
import math
import os
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from notebook_local import resolve_project_dir, print_local_setup

PROJECT_DIR = resolve_project_dir()
WORK_DIR = PROJECT_DIR / 'work'
NB02_DIR = WORK_DIR / 'notebook_02_bev_homography'
NB03A_DIR = WORK_DIR / 'notebook_03a_gt_annotations_bev_tracks'
NB03_OLD_DIR = WORK_DIR / 'notebook_03_bev_vehicles'
NB04_DIR = WORK_DIR / 'notebook_04_metric_scene_export'
NB04_DIR.mkdir(parents=True, exist_ok=True)

print_local_setup(PROJECT_DIR)
print('Notebook 02 directory:', NB02_DIR)
print('Notebook 03A directory:', NB03A_DIR)
print('Notebook 04 output directory:', NB04_DIR)

if not NB02_DIR.exists():
    raise FileNotFoundError('Notebook 02 output directory not found. Run Notebook 02 first: ' + str(NB02_DIR))
if not NB03A_DIR.exists():
    print('Warning: Notebook 03A output directory not found. Will try old Notebook 03 directory if needed.')


In [ ]:
#@title 2. Load homography from Notebook 02
homography_candidates = [
    NB02_DIR / 'homography_image_to_bev.json',
    NB02_DIR / 'bev_homography.json',
    NB02_DIR / 'homography.json',
]
homography_candidates += sorted(NB02_DIR.glob('*homography*.json'))

homography_path = None
for p in homography_candidates:
    if p.exists():
        homography_path = p
        break

if homography_path is None:
    print('JSON files found in Notebook 02 directory:')
    for p in sorted(NB02_DIR.glob('*.json')):
        print(' ', p)
    raise FileNotFoundError('Could not find a homography JSON file from Notebook 02.')

with open(homography_path, 'r') as f:
    homography_data = json.load(f)

print('Loaded homography JSON:', homography_path)
print('Homography keys:', list(homography_data.keys()))

if 'H_image_to_bev' in homography_data:
    H_image_to_bev = np.array(homography_data['H_image_to_bev'], dtype=np.float32)
elif 'homography' in homography_data:
    H_image_to_bev = np.array(homography_data['homography'], dtype=np.float32)
elif 'H' in homography_data:
    H_image_to_bev = np.array(homography_data['H'], dtype=np.float32)
else:
    raise KeyError('Homography JSON does not contain H_image_to_bev, homography, or H.')

BEV_WIDTH = int(homography_data.get('bev_width', homography_data.get('BEV_WIDTH', 600)))
BEV_HEIGHT = int(homography_data.get('bev_height', homography_data.get('BEV_HEIGHT', 1000)))

print('BEV size:', BEV_WIDTH, 'x', BEV_HEIGHT)
print('H_image_to_bev:')
print(H_image_to_bev)


In [ ]:
#@title 3. Load GT-based BEV tracks from Notebook 03A
track_candidates = []
if NB03A_DIR.exists():
    track_candidates += [
        NB03A_DIR / 'vehicle_tracks_bev.csv',
        NB03A_DIR / 'vehicle_tracks_bev_gt.csv',
    ]
    track_candidates += sorted(NB03A_DIR.glob('*bev*.csv'))

# Fallback to old Notebook 03 only if GT-based tracks do not exist.
if NB03_OLD_DIR.exists():
    track_candidates += [
        NB03_OLD_DIR / 'vehicle_tracks_bev.csv',
        NB03_OLD_DIR / 'projected_vehicle_tracks.csv',
        NB03_OLD_DIR / 'detections_projected_bev.csv',
    ]
    track_candidates += sorted(NB03_OLD_DIR.glob('*bev*.csv'))

tracks_path = None
for p in track_candidates:
    if p.exists():
        tracks_path = p
        break

if tracks_path is None:
    print('Candidate track files checked:')
    for p in track_candidates:
        print(' ', p)
    raise FileNotFoundError('Could not find vehicle_tracks_bev.csv from Notebook 03A.')

tracks = pd.read_csv(tracks_path)

print('Loaded tracks CSV:', tracks_path)
print('Shape:', tracks.shape)
print('Columns:', tracks.columns.tolist())
display(tracks.head())

required_any = ['frame_id', 'frame_idx', 'sample_frame_idx']
if not any(c in tracks.columns for c in required_any):
    raise KeyError('Tracks must contain one of these frame columns: ' + str(required_any))
if 'track_id' not in tracks.columns:
    raise KeyError('Tracks must contain track_id. If this is a YOLO-only detections CSV, rerun Notebook 03A.')
if not all(c in tracks.columns for c in ['x1', 'y1', 'x2', 'y2']):
    raise KeyError('Tracks must contain x1, y1, x2, y2 bounding box columns.')

# Normalize frame_id.
if 'frame_id' not in tracks.columns:
    if 'frame_idx' in tracks.columns:
        tracks['frame_id'] = tracks['frame_idx'].astype(int)
    else:
        tracks['frame_id'] = tracks['sample_frame_idx'].astype(int)

# Add class name if missing.
if 'class_name' not in tracks.columns:
    tracks['class_name'] = 'vehicle'

# If BEV coordinates are missing, compute them from bbox bottom-center.
if not all(c in tracks.columns for c in ['bev_x_px', 'bev_y_px']):
    print('BEV coordinates not found. Computing them from bbox bottom-center using homography.')
    cx = 0.5 * (pd.to_numeric(tracks['x1'], errors='coerce') + pd.to_numeric(tracks['x2'], errors='coerce'))
    cy = pd.to_numeric(tracks['y2'], errors='coerce')
    pts = np.stack([cx.to_numpy(dtype=np.float32), cy.to_numpy(dtype=np.float32)], axis=1).reshape(-1, 1, 2)
    out = cv2.perspectiveTransform(pts, H_image_to_bev).reshape(-1, 2)
    tracks['bev_x_px'] = out[:, 0]
    tracks['bev_y_px'] = out[:, 1]

print('Normalized track columns available:')
print(tracks.columns.tolist())
display(tracks.head())


## 4. Metric calibration

The homography produces BEV pixel coordinates. This section estimates pixels-per-meter. The default uses the median projected BEV width of annotated vehicle boxes and assumes an average car width of 1.8 m. This is approximate; lane-width calibration is better when you have reliable lane points.


In [ ]:
#@title 4A. Calibration settings
CALIBRATION_MODE = 'auto_vehicle_width_prior' #@param ['auto_vehicle_width_prior', 'manual_lane_width', 'manual_pixels_per_meter']
ASSUMED_CAR_WIDTH_M = 1.8 #@param {type:'number'}
LANE_POINT_A = [250, 520] #@param
LANE_POINT_B = [350, 520] #@param
ASSUMED_LANE_WIDTH_M = 3.5 #@param {type:'number'}
MANUAL_PIXELS_PER_METER = 25.0 #@param {type:'number'}

print('Calibration mode:', CALIBRATION_MODE)
print('Assumed car width meters:', ASSUMED_CAR_WIDTH_M)


In [ ]:
#@title 4B. Estimate pixels-per-meter
# This cell is robust to being run by itself after prior cells.
if 'CALIBRATION_MODE' not in globals():
    CALIBRATION_MODE = 'auto_vehicle_width_prior'
if 'ASSUMED_CAR_WIDTH_M' not in globals():
    ASSUMED_CAR_WIDTH_M = 1.8
if 'ASSUMED_LANE_WIDTH_M' not in globals():
    ASSUMED_LANE_WIDTH_M = 3.5
if 'MANUAL_PIXELS_PER_METER' not in globals():
    MANUAL_PIXELS_PER_METER = 25.0
if 'LANE_POINT_A' not in globals():
    LANE_POINT_A = [250, 520]
if 'LANE_POINT_B' not in globals():
    LANE_POINT_B = [350, 520]

def infer_vehicle_width_px_from_tracks(df, H):
    possible_width_cols = ['bev_box_width_px', 'box_width_bev_px', 'width_bev_px']
    for c in possible_width_cols:
        if c in df.columns:
            vals = pd.to_numeric(df[c], errors='coerce').dropna()
            vals = vals[(vals > 2) & (vals < 300)]
            if len(vals) >= 3:
                return float(vals.median()), c

    if all(c in df.columns for c in ['x1', 'x2', 'y2']):
        x1 = pd.to_numeric(df['x1'], errors='coerce').to_numpy(dtype=np.float32)
        x2 = pd.to_numeric(df['x2'], errors='coerce').to_numpy(dtype=np.float32)
        y2 = pd.to_numeric(df['y2'], errors='coerce').to_numpy(dtype=np.float32)
        pts_left = np.stack([x1, y2], axis=1).reshape(-1, 1, 2)
        pts_right = np.stack([x2, y2], axis=1).reshape(-1, 1, 2)
        bev_left = cv2.perspectiveTransform(pts_left, H).reshape(-1, 2)
        bev_right = cv2.perspectiveTransform(pts_right, H).reshape(-1, 2)
        widths = np.linalg.norm(bev_right - bev_left, axis=1)
        vals = pd.Series(widths).replace([np.inf, -np.inf], np.nan).dropna()
        vals = vals[(vals > 2) & (vals < 300)]
        if len(vals) >= 3:
            df['bev_box_width_px'] = widths
            return float(vals.median()), 'projected_bbox_bottom_width'

    if all(c in df.columns for c in ['x1', 'x2']):
        vals = (pd.to_numeric(df['x2'], errors='coerce') - pd.to_numeric(df['x1'], errors='coerce')).dropna().abs()
        vals = vals[(vals > 2) & (vals < 300)]
        if len(vals) >= 3:
            return float(vals.median()), 'image_bbox_width_fallback'

    return 45.0, 'fallback_constant'

if CALIBRATION_MODE == 'auto_vehicle_width_prior':
    median_width_px, source_col = infer_vehicle_width_px_from_tracks(tracks, H_image_to_bev)
    pixels_per_meter = median_width_px / ASSUMED_CAR_WIDTH_M
    calibration_details = {
        'mode': CALIBRATION_MODE,
        'assumed_car_width_m': ASSUMED_CAR_WIDTH_M,
        'median_width_px': median_width_px,
        'width_source': source_col,
    }
elif CALIBRATION_MODE == 'manual_lane_width':
    a = np.array(LANE_POINT_A, dtype=float)
    b = np.array(LANE_POINT_B, dtype=float)
    lane_width_px = float(np.linalg.norm(a - b))
    pixels_per_meter = lane_width_px / ASSUMED_LANE_WIDTH_M
    calibration_details = {
        'mode': CALIBRATION_MODE,
        'lane_point_a': LANE_POINT_A,
        'lane_point_b': LANE_POINT_B,
        'assumed_lane_width_m': ASSUMED_LANE_WIDTH_M,
        'lane_width_px': lane_width_px,
    }
elif CALIBRATION_MODE == 'manual_pixels_per_meter':
    pixels_per_meter = float(MANUAL_PIXELS_PER_METER)
    calibration_details = {
        'mode': CALIBRATION_MODE,
        'manual_pixels_per_meter': pixels_per_meter,
    }
else:
    raise ValueError('Unknown CALIBRATION_MODE: ' + str(CALIBRATION_MODE))

meters_per_pixel = 1.0 / pixels_per_meter
print('Pixels per meter:', pixels_per_meter)
print('Meters per pixel:', meters_per_pixel)
print('Calibration details:', calibration_details)

calibration_json = {
    'pixels_per_meter': pixels_per_meter,
    'meters_per_pixel': meters_per_pixel,
    'bev_width_px': BEV_WIDTH,
    'bev_height_px': BEV_HEIGHT,
    'details': calibration_details,
}
calibration_path = NB04_DIR / 'metric_calibration.json'
with open(calibration_path, 'w') as f:
    json.dump(calibration_json, f, indent=2)
print('Saved calibration:', calibration_path)


In [ ]:
#@title 5. Convert BEV tracks from pixels to meters
metric = tracks.copy()
metric['bev_x_px'] = pd.to_numeric(metric['bev_x_px'], errors='coerce')
metric['bev_y_px'] = pd.to_numeric(metric['bev_y_px'], errors='coerce')
metric['frame_id'] = pd.to_numeric(metric['frame_id'], errors='coerce').astype(int)
metric['track_id'] = pd.to_numeric(metric['track_id'], errors='coerce').astype(int)
metric = metric.dropna(subset=['bev_x_px', 'bev_y_px']).copy()

metric['x_m'] = metric['bev_x_px'] * meters_per_pixel
metric['y_m'] = (BEV_HEIGHT - metric['bev_y_px']) * meters_per_pixel

FPS = 8.0 #@param {type:'number'}
metric['t_s'] = metric['frame_id'] / FPS

metric = metric.sort_values(['track_id', 'frame_id']).reset_index(drop=True)
metric['vx_mps'] = 0.0
metric['vy_mps'] = 0.0
metric['speed_mps'] = 0.0
metric['yaw_rad'] = 0.0

for track_id, group in metric.groupby('track_id'):
    idx = group.index.to_numpy()
    xs = group['x_m'].to_numpy(dtype=float)
    ys = group['y_m'].to_numpy(dtype=float)
    ts = group['t_s'].to_numpy(dtype=float)

    if len(idx) == 1:
        continue

    vx = np.zeros(len(idx), dtype=float)
    vy = np.zeros(len(idx), dtype=float)
    yaw = np.zeros(len(idx), dtype=float)
    speed = np.zeros(len(idx), dtype=float)

    for i in range(len(idx)):
        if i == 0:
            j0, j1 = 0, 1
        elif i == len(idx) - 1:
            j0, j1 = len(idx) - 2, len(idx) - 1
        else:
            j0, j1 = i - 1, i + 1
        dt = max(ts[j1] - ts[j0], 1e-6)
        dx = xs[j1] - xs[j0]
        dy = ys[j1] - ys[j0]
        vx[i] = dx / dt
        vy[i] = dy / dt
        speed[i] = math.hypot(vx[i], vy[i])
        yaw[i] = math.atan2(vy[i], vx[i])

    metric.loc[idx, 'vx_mps'] = vx
    metric.loc[idx, 'vy_mps'] = vy
    metric.loc[idx, 'speed_mps'] = speed
    metric.loc[idx, 'yaw_rad'] = yaw

metric_path = NB04_DIR / 'vehicle_tracks_metric.csv'
metric.to_csv(metric_path, index=False)
print('Saved metric tracks:', metric_path)
print('Rows:', len(metric))
print('Unique tracks:', metric['track_id'].nunique())
display(metric.head())


In [ ]:
#@title 6. Build metric 3D vehicle cuboids
DEFAULT_DIMS = {
    'car': (4.5, 1.8, 1.5),
    'truck': (8.0, 2.5, 3.0),
    'bus': (12.0, 2.6, 3.2),
    'motorcycle': (2.2, 0.8, 1.4),
    'bicycle': (1.8, 0.6, 1.4),
    'vehicle': (4.5, 1.8, 1.5),
    'unknown': (4.5, 1.8, 1.5),
}

def normalize_class_name_value(value):
    name = str(value).lower() if pd.notna(value) else 'vehicle'
    if 'bus' in name:
        return 'bus'
    if 'truck' in name:
        return 'truck'
    if 'motor' in name:
        return 'motorcycle'
    if 'bike' in name or 'bicycle' in name:
        return 'bicycle'
    if 'car' in name:
        return 'car'
    if 'vehicle' in name:
        return 'vehicle'
    return 'vehicle'

cuboids = []
for _, row in metric.iterrows():
    cls = normalize_class_name_value(row.get('class_name', 'vehicle'))
    length_m, width_m, height_m = DEFAULT_DIMS.get(cls, DEFAULT_DIMS['unknown'])
    cuboids.append({
        'frame_id': int(row['frame_id']),
        'track_id': int(row['track_id']),
        'class_name': cls,
        'center_x_m': float(row['x_m']),
        'center_y_m': float(row['y_m']),
        'center_z_m': height_m / 2.0,
        'length_m': length_m,
        'width_m': width_m,
        'height_m': height_m,
        'yaw_rad': float(row['yaw_rad']) if np.isfinite(row['yaw_rad']) else 0.0,
        'speed_mps': float(row['speed_mps']) if np.isfinite(row['speed_mps']) else 0.0,
    })

cuboids_df = pd.DataFrame(cuboids)
cuboids_path = NB04_DIR / 'vehicle_cuboids_metric.csv'
cuboids_df.to_csv(cuboids_path, index=False)

print('Saved cuboids:', cuboids_path)
print('Rows:', len(cuboids_df))
print('Unique tracks:', cuboids_df['track_id'].nunique() if len(cuboids_df) else 0)
display(cuboids_df.head())


In [ ]:
#@title 7. Plot metric BEV tracks
fig_width = 8
fig_height = fig_width * BEV_HEIGHT / BEV_WIDTH
fig, ax = plt.subplots(figsize=(fig_width, fig_height))

for track_id, group in metric.groupby('track_id'):
    group = group.sort_values('frame_id')
    ax.plot(group['x_m'], group['y_m'], marker='o', linewidth=1, markersize=3)
    last = group.iloc[-1]
    ax.text(last['x_m'], last['y_m'], str(track_id), fontsize=8)

ax.set_title('Metric BEV vehicle tracks')
ax.set_xlabel('X meters')
ax.set_ylabel('Y meters')
ax.set_aspect('equal', adjustable='box')
ax.grid(True)

bev_metric_overlay_path = NB04_DIR / 'metric_bev_tracks.png'
plt.savefig(bev_metric_overlay_path, dpi=160, bbox_inches='tight')
plt.show()
print('Saved metric BEV tracks:', bev_metric_overlay_path)


In [ ]:
#@title 8. 3D preview with metric cuboids
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

def cuboid_vertices(cx, cy, cz, length, width, height, yaw):
    x = length / 2.0
    y = width / 2.0
    z = height / 2.0
    local = np.array([
        [-x, -y, -z], [ x, -y, -z], [ x,  y, -z], [-x,  y, -z],
        [-x, -y,  z], [ x, -y,  z], [ x,  y,  z], [-x,  y,  z],
    ], dtype=float)
    c = math.cos(yaw)
    s = math.sin(yaw)
    R = np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]], dtype=float)
    return local @ R.T + np.array([cx, cy, cz])

def cuboid_faces(verts):
    return [
        [verts[i] for i in [0, 1, 2, 3]],
        [verts[i] for i in [4, 5, 6, 7]],
        [verts[i] for i in [0, 1, 5, 4]],
        [verts[i] for i in [1, 2, 6, 5]],
        [verts[i] for i in [2, 3, 7, 6]],
        [verts[i] for i in [3, 0, 4, 7]],
    ]

PREVIEW_FRAME_ID = int(cuboids_df['frame_id'].min()) #@param {type:'integer'}
frame_df = cuboids_df[cuboids_df['frame_id'] == PREVIEW_FRAME_ID].copy()
if len(frame_df) == 0:
    raise ValueError('No cuboids for PREVIEW_FRAME_ID: ' + str(PREVIEW_FRAME_ID))

fig = plt.figure(figsize=(9, 9))
ax = fig.add_subplot(111, projection='3d')
road_w_m = BEV_WIDTH * meters_per_pixel
road_h_m = BEV_HEIGHT * meters_per_pixel
ax.plot([0, road_w_m, road_w_m, 0, 0], [0, 0, road_h_m, road_h_m, 0], [0, 0, 0, 0, 0], linewidth=2)

for _, row in frame_df.iterrows():
    verts = cuboid_vertices(row['center_x_m'], row['center_y_m'], row['center_z_m'], row['length_m'], row['width_m'], row['height_m'], row['yaw_rad'])
    faces = cuboid_faces(verts)
    poly = Poly3DCollection(faces, alpha=0.35, linewidths=0.6, edgecolors='k')
    ax.add_collection3d(poly)
    ax.text(row['center_x_m'], row['center_y_m'], row['center_z_m'] + row['height_m'], str(int(row['track_id'])), fontsize=8)

ax.set_xlim(0, road_w_m)
ax.set_ylim(0, road_h_m)
ax.set_zlim(0, 8)
ax.set_xlabel('X meters')
ax.set_ylabel('Y meters')
ax.set_zlabel('Z meters')
ax.set_title('Metric 3D cuboids, frame ' + str(PREVIEW_FRAME_ID))
ax.view_init(elev=45, azim=-60)
ax.set_box_aspect((road_w_m, road_h_m, 10))
plt.tight_layout()
preview_path = NB04_DIR / 'metric_3d_cuboids_preview.png'
plt.savefig(preview_path, dpi=160, bbox_inches='tight')
plt.show()
print('Saved 3D preview:', preview_path)


In [ ]:
#@title 9. Export simple OBJ scene for one frame
OBJ_FRAME_ID = PREVIEW_FRAME_ID #@param {type:'integer'}
obj_df = cuboids_df[cuboids_df['frame_id'] == OBJ_FRAME_ID].copy()
if len(obj_df) == 0:
    raise ValueError('No cuboids for OBJ_FRAME_ID: ' + str(OBJ_FRAME_ID))

obj_path = NB04_DIR / ('scene_frame_' + str(OBJ_FRAME_ID).zfill(6) + '.obj')
vertices = []
faces = []
road_w_m = BEV_WIDTH * meters_per_pixel
road_h_m = BEV_HEIGHT * meters_per_pixel
road_start = len(vertices) + 1
vertices.extend([(0, 0, 0), (road_w_m, 0, 0), (road_w_m, road_h_m, 0), (0, road_h_m, 0)])
faces.append((road_start, road_start + 1, road_start + 2, road_start + 3))

for _, row in obj_df.iterrows():
    verts = cuboid_vertices(row['center_x_m'], row['center_y_m'], row['center_z_m'], row['length_m'], row['width_m'], row['height_m'], row['yaw_rad'])
    start = len(vertices) + 1
    vertices.extend([tuple(v) for v in verts])
    for face_ids in [[0,1,2,3], [4,5,6,7], [0,1,5,4], [1,2,6,5], [2,3,7,6], [3,0,4,7]]:
        faces.append(tuple(start + i for i in face_ids))

nl = chr(10)
with open(obj_path, 'w') as f:
    f.write('# UAVDT metric cuboid scene' + nl)
    for v in vertices:
        f.write('v ' + str(v[0]) + ' ' + str(v[1]) + ' ' + str(v[2]) + nl)
    for face in faces:
        f.write('f ' + ' '.join(str(i) for i in face) + nl)

print('Saved OBJ scene:', obj_path)
print('Vertices:', len(vertices))
print('Faces:', len(faces))


In [ ]:
#@title 10. Export dynamic scene JSON
scene_json = {
    'coordinate_system': {
        'x': 'right in BEV, meters',
        'y': 'forward/up in BEV, meters',
        'z': 'up, meters',
    },
    'calibration': calibration_json,
    'source_files': {
        'homography_json': str(homography_path),
        'tracks_csv': str(tracks_path),
    },
    'vehicle_dimensions_default_m': {
        k: {'length': v[0], 'width': v[1], 'height': v[2]} for k, v in DEFAULT_DIMS.items()
    },
    'frames': [],
}

for frame_id, group in cuboids_df.groupby('frame_id'):
    frame_record = {
        'frame_id': int(frame_id),
        'time_s': float(frame_id / FPS),
        'vehicles': [],
    }
    for _, row in group.iterrows():
        frame_record['vehicles'].append({
            'track_id': int(row['track_id']),
            'class_name': row['class_name'],
            'center_m': [float(row['center_x_m']), float(row['center_y_m']), float(row['center_z_m'])],
            'size_m': [float(row['length_m']), float(row['width_m']), float(row['height_m'])],
            'yaw_rad': float(row['yaw_rad']),
            'speed_mps': float(row['speed_mps']),
        })
    scene_json['frames'].append(frame_record)

scene_json_path = NB04_DIR / 'dynamic_metric_scene.json'
with open(scene_json_path, 'w') as f:
    json.dump(scene_json, f, indent=2)

print('Saved dynamic metric scene:', scene_json_path)
print('Frames:', len(scene_json['frames']))
print('Objects:', sum(len(fr['vehicles']) for fr in scene_json['frames']))


## Next

After this notebook, run Notebook 05 to create dynamic videos and interactive visualizations from:

- `vehicle_tracks_metric.csv`
- `vehicle_cuboids_metric.csv`
- `dynamic_metric_scene.json`
